# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the **FAIR^2** dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` values as per the Croissant schema.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets and fields using their `@id` values.

> **Note:** To list record sets and their fields, all references use the entity's `@id` for clarity and reproducibility.

In [ ]:
# List available record sets and their fields (by @id)
record_sets = [rs for rs in dataset.record_sets]
print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs['name']}")
    if 'fields' in rs:
        print("  Fields:")
        for f in rs['fields']:
            print(f"    - @id: {f['@id']} | name: {f['name']} | dataType: {f.get('dataType', 'Unknown')}")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for further analysis, referencing all record set and field IDs.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = dict()

for record_set_id in record_set_ids:
    # Extract records using the record set @id
    records = list(dataset.records(record_set=record_set_id))
    # Only create a DataFrame if records exist
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

if len(dataframes) == 0:
    print('No records found in any record set.')
else:
    # Print available dataframes and column names
    for rid, df in dataframes.items():
        print(f'---\nRecord set @id: {rid}')
        print(f'Columns: {list(df.columns)}\nSample rows:')
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply exploration and data preparation steps using field `@id`s. We select a numerical field from one of the dataframes, filter by a threshold, normalize it, and group by a categorical field (if available).

In [ ]:
# For demonstration, pick the first available dataframe
if len(dataframes) == 0:
    print('No tabular data available for EDA.')
else:
    # Select the first record set for EDA
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    print(f'Working with record set: {record_set_id}')

    # Attempt to automatically find a numeric column to use
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print('No numeric field found for analysis.')
    else:
        threshold = df[numeric_field_id].mean()

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} (@id) > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) 
            / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} data:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to find a categorical/groupable field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and (df[col].dtype == object):
                group_field_id = col
                break
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean').reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print('No suitable categorical field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All field references use their `@id`.

_Example: Histogram of a numeric field and bar plot of group means._

In [ ]:
import matplotlib.pyplot as plt

if len(dataframes) == 0 or numeric_field_id is None:
    print('No data available for visualization.')
else:
    # Histogram for the numeric field
    plt.figure(figsize=(6, 4))
    df[numeric_field_id].hist(bins=12, alpha=0.7)
    plt.title(f'Histogram of {numeric_field_id} (@id)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.3)
    plt.show()

    # Bar plot for grouped means (if grouping field available)
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(7, 4))
        plt.bar(grouped_df[group_field_id].astype(str), grouped_df['mean'], alpha=0.8)
        plt.title(f'Mean {numeric_field_id} by {group_field_id} (@id)')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 colorectal cancer survivors dataset using the `mlcroissant` library. We extracted metadata and tabular data using explicit Croissant `@id` values for reproducibility, performed filtering and normalization on numeric fields, and visualized the results. Further analysis may include detailed modeling or domain-specific statistical investigation using the available fields and values, always referencing their Croissant `@id`s for clarity and traceability.